# Tự động cào dữ liệu sản phẩm từ Tiki API và tạo file SQL seeding

Notebook này tự động lấy sản phẩm từ Tiki cho tất cả 25 danh mục con trong database, tạo dữ liệu đấu giá ngẫu nhiên và lưu vào file `product.insert.sql`.

In [ ]:
import os
import json
import random
import requests
import psycopg2
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv()
print("✅ Thư viện đã được import")

In [ ]:
TIKI_CATEGORY_URL = "https://tiki.vn/api/personalish/v1/blocks/listings"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Ánh xạ tên danh mục trong database sang ID danh mục tương ứng của Tiki API
DB_CAT_TO_TIKI = {
    "Điện Thoại": 1795,
    "Laptop": 8095,
    "Máy Tính Bảng": 1794,
    "Phụ Kiện Điện Tử": 1815,
    "Thiết Bị Âm Thanh": 1801,
    "Áo Nam": 925,
    "Áo Nữ": 5404,
    "Quần Jeans": 925,
    "Giày Dép": 27572,
    "Phụ Kiện Thời Trang": 8374,
    "Bàn Ghế": 8374,
    "Sofa": 8374,
    "Đồ Dùng Nhà Bếp": 1815,
    "Đèn Trang Trí": 8374,
    "Chăn Ga Gối": 5404,
    "Dụng Cụ Tập Gym": 27572,
    "Xe Đạp": 1801,
    "Giày Thể Thao": 27572,
    "Phụ Kiện Du Lịch": 925,
    "Dụng Cụ Camping": 1815,
    "Sách Văn Học": 27224,
    "Sách Kỹ Năng": 27224,
    "Văn Phòng Phẩm": 1815,
    "Dụng Cụ Học Tập": 1815,
    "Sổ Tay": 27224
}

def get_db_connection():
    conn_str = os.getenv("SUPABASE_CONNECTION_STRING")
    if conn_str:
        return psycopg2.connect(conn_str)
    return psycopg2.connect(
        host=os.getenv("DB_HOST", "localhost"),
        port=os.getenv("DB_PORT", "5432"),
        user=os.getenv("DB_USER", "postgres"),
        password=os.getenv("DB_PASSWORD", "my_local_password"),
        database=os.getenv("DB_NAME", "online_auction_test")
    )

def generate_start_time():
    now = datetime.now()
    days_offset = random.randint(-3, 3)
    hours_offset = random.randint(0, 23)
    return now + timedelta(days=days_offset, hours=hours_offset)

def generate_end_time(start_time):
    days_to_add = random.randint(1, 20)
    return start_time + timedelta(days=days_to_add)

def generate_prices(base_price):
    if not base_price or base_price <= 0:
        base_price = random.randint(50, 500) * 10000
    start_price = int(base_price * random.uniform(0.6, 0.9))
    step_price = int(start_price * 0.05)
    buy_now_price = int(start_price * 1.5)
    return {
        'start_price': start_price,
        'current_price': start_price,
        'step_price': step_price,
        'buy_now_price': buy_now_price
    }

In [ ]:
print("Connecting to database...")
conn = get_db_connection()
cursor = conn.cursor()

# Lấy danh sách subcategories từ database
cursor.execute("SELECT id, name FROM categories WHERE parent_id IS NOT NULL")
db_categories = cursor.fetchall()

# Lấy danh sách user_ids để làm seller_id
cursor.execute("SELECT user_id FROM users")
seller_ids = [row[0] for row in cursor.fetchall()]
if not seller_ids:
    seller_ids = [1]

products_data = []
print(f"Bắt đầu cào dữ liệu cho {len(db_categories)} danh mục con...")

for cat_id, cat_name in db_categories:
    tiki_cat_id = DB_CAT_TO_TIKI.get(cat_name, 1815)
    # Mỗi danh mục lấy ngẫu nhiên từ 20 đến 50 sản phẩm
    target_count = random.randint(20, 50)
    
    print(f"-> Lấy {target_count} sản phẩm cho '{cat_name}' (Tiki ID: {tiki_cat_id})")
    
    try:
        response = requests.get(
            TIKI_CATEGORY_URL,
            headers=headers,
            params={'limit': target_count, 'category': tiki_cat_id, 'page': 1}
        )
        
        if response.status_code == 200:
            tiki_products = response.json().get('data', [])
            for product in tiki_products:
                name = product.get('name', '')
                thumbnail_url = product.get('thumbnail_url', '')
                price = product.get('price', 0)
                
                name_escaped = name.replace("'", "''")
                desc = f"Sản phẩm {name} chất lượng cao. Hàng chính hãng, uy tín và đảm bảo chất lượng.".replace("'", "''")
                
                images = [thumbnail_url, thumbnail_url, thumbnail_url]
                images_array = "ARRAY[" + ", ".join([f"'{img}'" for img in images]) + "]"
                
                prices = generate_prices(price)
                start_time = generate_start_time()
                end_time = generate_end_time(start_time)
                seller_id = random.choice(seller_ids)
                auto_ext = str(random.choice([True, False])).lower()
                
                sql = f"""INSERT INTO products (product_name, seller_id, step_price, start_price, current_price, buy_now_price, start_time, end_time, cat2_id, description, product_images, auto_extended)
VALUES ('{name_escaped}', {seller_id}, {prices['step_price']}, {prices['start_price']}, {prices['current_price']}, {prices['buy_now_price']}, '{start_time.isoformat()}', '{end_time.isoformat()}', {cat_id}, '{desc}', {images_array}, {auto_ext});"""
                products_data.append(sql)
        else:
            print(f"   ❌ Lỗi kết nối Tiki: {response.status_code}")
    except Exception as e:
        print(f"   ❌ Lỗi xử lý: {e}")

# Ghi vào file SQL
with open('product.insert.sql', 'w', encoding='utf-8') as f:
    f.write('\n'.join(products_data))

print(f"\n🎉 Đã tạo thành công file product.insert.sql với {len(products_data)} sản phẩm!")
cursor.close()
conn.close()

In [ ]:
# Thực thi file SQL để nạp dữ liệu trực tiếp vào database hiện tại
print("Bắt đầu nạp sản phẩm vào Database...")
with open('product.insert.sql', 'r', encoding='utf-8') as f:
    sql_content = f.read()

conn = get_db_connection()
cursor = conn.cursor()
try:
    cursor.execute(sql_content)
    conn.commit()
    print("✅ Đã nạp thành công toàn bộ sản phẩm vào Database!")
except Exception as e:
    conn.rollback()
    print(f"❌ Lỗi khi nạp dữ liệu: {e}")
finally:
    cursor.close()
    conn.close()